# Introduction of the Q.ANT SDK Python API

## Overview

This notebook provides an introduction to the **Q.ANT Native Computing Toolkit Python API**, demonstrating how to use various mathematical and AI operations available in the SDK.

### Purpose of this Notebook

This notebook demonstrates:
- **Mathematical operations** such as element-wise multiplication and scaled periodic functions
- **AI/Neural network operations** including linear layers, convolutions, (learnable) activation functions, pooling and normalization
- **Validation** of Q.ANT SDK operations against PyTorch reference implementations using bfloat16 precision

### Structure

Each section:
1. Creates random test data in bfloat16 format
2. Computes results using both Q.ANT SDK and PyTorch
3. Compares the outputs to validate correctness
4. Records error metrics for testing purposes

### Prerequisites
Install the requirements listed in the requirements.txt file in the folder.
```bash
pip intall -r requirements.txt
```

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qant_native_computing_toolkit.ai as q_ai
import qant_native_computing_toolkit.native as q_native
import torch
from ml_dtypes import bfloat16

We fix random seeds for reproducibility and initialize a dictionary to store error metrics for automated testing.

In [ ]:
# fix random seeds
np.random.seed(0)
torch.random.manual_seed(0)

error_dict_for_test = {}

This helper function compares Q.ANT SDK results with PyTorch reference implementations, calculating:
- **Relative error**: Mean ratio of absolute error to reference values
- **Absolute error**: Mean absolute difference between results
- **Max absolute error**: Largest absolute difference observed

An optional histogram visualization shows the error distribution.

In [ ]:
def print_errors(r, r_torch, plot_histogram=False):
    absolute_error = np.abs(r - r_torch)
    print(
        "Relative error:",
        (absolute_error / np.abs(r_torch)).mean()
        if np.abs(r_torch).mean() > 0
        else "N/A",
    )
    print("Absolute error:", absolute_error.mean())
    print("Max absolute error:", absolute_error.max())

    if plot_histogram:
        # Plot error distribution
        plt.figure(figsize=(10, 5))
        plt.hist(
            absolute_error.astype(np.float32).flatten(),
            bins=50,
            alpha=0.7,
            color="blue",
        )
        plt.title("Absolute Error Distribution")
        plt.xlabel("Error")
        plt.ylabel("Frequency")
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    return absolute_error.tolist()

# Mathematical Operations

This section demonstrates basic mathematical operations provided by the Q.ANT SDK.

## Element-wise (Hadamard) Multiplication

Element-wise (Hadamard) multiplication multiplies corresponding elements of two arrays. This is a fundamental operation used throughout machine learning, particularly in attention mechanisms and gating operations.

**Example**: For arrays `x` and `y`, the result `r[i,j] = x[i,j] * y[i,j]`

In [ ]:
x = np.random.rand(100, 10).astype(bfloat16)
y = np.random.rand(100, 10).astype(bfloat16)

# calculate the reference elementwise product
e = x * y

# compute the elementwise product using the qant SDK
r = q_native.mul_elementwise(x, y)


err = print_errors(r, e)
error_dict_for_test["mul_elementwise"] = err

## Scaled Sinusoidal

This operation computes an approximated cosine ($\tilde{\cos}$) with scaled amplitude:
$$f(w, x) = w \cdot \tilde{\cos}(x)$$
This is useful for periodic activations and various sinusoidal operations.

In [ ]:
x = np.random.rand(1024).astype(bfloat16)
w = np.random.rand(1024).astype(bfloat16)

r = q_native.calc_scaled_periodic_nl_fprop(x, w)
e = w * np.cos(x)

err = print_errors(r, e)
error_dict_for_test["calc_scaled_periodic_nl_fprop"] = err

### Visualization of the approximated cosine

In [ ]:
# Compare Q.ANT approximation against the original cosine-based reference
times = np.linspace(0, 2, num=1024).astype(bfloat16)
phases = (2 * np.pi * times).astype(bfloat16)
amplitudes = np.ones_like(phases).astype(bfloat16)

approx_cos = q_native.calc_scaled_periodic_nl_fprop(phases, amplitudes)
torch_cos = amplitudes * np.cos(phases)

# Signal comparison
plt.figure()
plt.plot(times, torch_cos, label="true cosine", linewidth=2, alpha=0.9)
plt.plot(times, approx_cos, label="approximated cosine", linewidth=1, alpha=0.85)
plt.title("Approximated Cosine vs Original")
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# AI Operations

This section demonstrates neural network operations commonly used in deep learning models. All operations are validated against PyTorch implementations to ensure correctness.

## Linear Layer

A linear layer performs matrix multiplication: `output = input @ weight.T`

This is the fundamental building block of neural networks, transforming input features through learned weight matrices.

In [ ]:
# batch size 128, input features 32, output features 64
weight = np.random.rand(64, 32).astype(bfloat16)
input = np.random.rand(128, 32).astype(bfloat16)

# calculate the reference with PyTorch
input_torch = torch.from_numpy(input.astype(np.float32)).to(torch.bfloat16)
weight_torch = torch.from_numpy(weight.astype(np.float32)).to(torch.bfloat16)
r_torch = (
    torch.nn.functional.linear(input_torch, weight_torch)
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)

r = q_ai.linear_fprop(input, weight)

err = print_errors(r, r_torch)
error_dict_for_test["linear_fprop"] = err

## 2D Convolution Layer

Convolutional layers are essential for processing spatial data like images. They apply learned filters (kernels) across the input to detect features like edges, textures, and patterns.

**Parameters demonstrated**:
- `stride`: Step size when moving the kernel
- `padding`: Zero-padding around input borders
- `dilation`: Spacing between kernel elements

In [ ]:
kernel_size = 3
input = np.random.rand(1, 3, 128, 128).astype(bfloat16)
weight = np.random.rand(1, 3, kernel_size, kernel_size).astype(bfloat16)

r = q_ai.conv_fprop(input, weight, stride=1, padding=1, dilation=1)

input_torch = torch.from_numpy(input.astype(np.float32)).to(torch.bfloat16)
weight_torch = torch.from_numpy(weight.astype(np.float32)).to(torch.bfloat16)
r_torch = (
    torch.nn.functional.conv2d(
        input_torch, weight_torch, stride=1, padding=1, dilation=1
    )
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)

err = print_errors(r, r_torch)
error_dict_for_test["conv_fprop"] = err

## 2D Transposed Convolution Layer

Transposed convolution (also called deconvolution) performs upsampling, increasing spatial dimensions. It's commonly used in:
- Decoder networks (e.g., autoencoders, U-Net)
- Generative models (e.g., GANs)
- Image super-resolution

**Note**: The weight tensor format differs between Q.ANT SDK and PyTorch, requiring axis swapping for validation.

In [ ]:
kernel_size = 3
input = np.random.rand(1, 3, 128, 128).astype(bfloat16)
# Note: For conv_transpose, the weight shape is (in_channels, out_channels, kernel_size, kernel_size)
weight = np.random.rand(3, 1, kernel_size, kernel_size).astype(bfloat16)

r = q_ai.conv_transpose_fprop(
    input, weight, stride=1, padding=1, dilation=1, output_padding=0
)

input_torch = torch.from_numpy(input.astype(np.float32)).to(torch.bfloat16)
weight_torch = torch.from_numpy(weight.astype(np.float32)).to(torch.bfloat16)
r_torch = (
    torch.nn.functional.conv_transpose2d(
        input_torch, weight_torch, stride=1, padding=1, dilation=1, output_padding=0
    )
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)

err = print_errors(r, r_torch)
error_dict_for_test["conv_transpose_fprop"] = err

## Bias

Every linear and convolutional layer typically includes a bias term: `output = activation(weights @ input + bias)`

This allows the model to learn non-zero baseline values for each feature.

In [ ]:
input = np.random.rand(128, 32).astype(bfloat16)
bias = np.random.rand(32).astype(bfloat16)

r = q_ai.add_bias_fprop(input, bias)
r_torch = (
    (
        torch.from_numpy(input.astype(np.float32)).to(torch.bfloat16)
        + torch.from_numpy(bias.astype(np.float32)).to(torch.bfloat16)
    )
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)

err = print_errors(r, r_torch)
error_dict_for_test["add_bias_fprop"] = err

## Activation Functions

Activation functions introduce non-linearity into neural networks, enabling them to learn complex patterns.

**ReLU (Rectified Linear Unit)**: `max(0, x)` - Most common activation, computationally efficient

**Sigmoid**: `1 / (1 + e^(-x))` - Squashes values to (0, 1), used in binary classification

**Softmax**: Normalizes values to a probability distribution (sums to 1), used in multi-class classification

In [ ]:
input = np.random.rand(128, 32).astype(bfloat16)

# ReLU activation function
r = q_ai.relu_fprop(input)
r_torch = (
    torch.nn.functional.relu(
        torch.from_numpy(input.astype(np.float32)).to(torch.bfloat16)
    )
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)
err = print_errors(r, r_torch)
error_dict_for_test["relu_fprop"] = err


# Sigmoid activation function
r = q_ai.sigmoid_fprop(input)
r_torch = (
    torch.nn.functional.sigmoid(
        torch.from_numpy(input.astype(np.float32)).to(torch.bfloat16)
    )
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)
err = print_errors(r, r_torch)
error_dict_for_test["sigmoid_fprop"] = err

# Softmax activation function
r = q_ai.softmax_fprop(input[0])
r_torch = (
    torch.nn.functional.softmax(
        torch.from_numpy(input.astype(np.float32)).to(torch.bfloat16), dim=-1
    )
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)
err = print_errors(r, r_torch)
error_dict_for_test["softmax_fprop"] = err

## Learnable Activation Function (KAN Layer)

Traditional activation functions (ReLU, Sigmoid, etc.) are fixed non-linear functions. **Kolmogorov-Arnold Network (KAN)** layers use learnable activation functions.

### How it Works

Instead of fixed activation functions, a sum of cosine-like basis functions ($\tilde{\cos}$) with learnable parameters are used. The result is a KAN layer that can be used in Fourier-based neural networks:

$$y_j = \sum_{i, l} a_{jil} \cdot \tilde{\cos}(k_l \cdot x_i + \phi_{jil})$$

Where:
- $x_i$: Input channels
- $y_j$: Output channels  
- $k_l$: Frequency parameters (typically integers: 1, 2, 3, ...)
- $\phi_{jil}$: Phase shifts (learnable)
- $a_{jil}$: Amplitudes (learnable)

In [ ]:
n_channels_in = 16
n_channels_out = 32
n_ks = 4

input = np.random.rand(n_channels_in).astype(bfloat16)
phis = np.random.rand(n_channels_out, n_channels_in, n_ks).astype(bfloat16)
amplitudes = np.random.rand(n_channels_out, n_channels_in, n_ks).astype(bfloat16)
ks = np.arange(1, n_ks + 1).astype(bfloat16)

r = q_ai.calc_kan_layer_fprop(input, phis, amplitudes, ks)
e = np.sum(
    amplitudes * np.cos(input[None, :, None] * ks[None, None, :] + phis), axis=(1, 2)
)

err = print_errors(r, e)
error_dict_for_test["calc_kan_layer_fprop"] = err

## Pooling Layers

Pooling layers reduce spatial dimensions while retaining important features, making networks more computationally efficient and robust to small translations.

**Max. Pooling 2D**: Selects the maximum value in each pooling window - preserves strong activations

**Average Pooling 2D**: Averages values in each pooling window - provides smoother downsampling

**Adaptive Max. Pooling 2D**: Selects the maximum value in each pooling window and automatically calculates pooling parameters to achieve a target output size.

**Adaptive Average Pooling 2D**: Averages values in each pooling window and automatically calculates pooling parameters to achieve a target output size.

In [ ]:
input = np.random.rand(3, 128, 128).astype(bfloat16)
input_torch = torch.from_numpy(input.astype(np.float32)).to(torch.bfloat16)

# MaxPool2D
r = q_ai.maxpool2d_fprop(input, kernel_size=2, stride=2, padding=0)
r_torch = (
    torch.nn.functional.max_pool2d(input_torch, kernel_size=2, stride=2, padding=0)
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)
print("MaxPool2D:")
err = print_errors(r, r_torch)
error_dict_for_test["maxpool2d_fprop"] = err

# AvgPool2D
r = q_ai.avgpool2d_fprop(input, kernel_size=2, stride=2, padding=0)
r_torch = (
    torch.nn.functional.avg_pool2d(input_torch, kernel_size=2, stride=2, padding=0)
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)
print("AveragePool2D:")
err = print_errors(r, r_torch)
error_dict_for_test["avgpool2d_fprop"] = err

# AdaptiveMaxPool2D
output_size = (2, 2)
r = q_ai.adaptive_maxpool2d_fprop(input, output_size=output_size)
r_torch = (
    torch.nn.functional.adaptive_max_pool2d(input_torch, output_size=output_size)
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)
print("AdaptiveMaxPool2D:")
err = print_errors(r, r_torch)
error_dict_for_test["adaptive_maxpool2d_fprop"] = err

# AdaptiveAvgPool2D
output_size = (2, 2)
r = q_ai.adaptive_avgpool2d_fprop(input, output_size=output_size)
r_torch = (
    torch.nn.functional.adaptive_avg_pool2d(input_torch, output_size=output_size)
    .to(torch.float32)
    .numpy()
    .astype(bfloat16)
)
print("AdaptiveAvgPool2D:")
err = print_errors(r, r_torch)
error_dict_for_test["adaptive_avgpool2d_fprop"] = err

## Normalization Layer

Batch normalization normalizes layer inputs, stabilizing training and enabling higher learning rates. It:
1. Normalizes inputs using batch statistics (mean and variance)
2. Applies learned scale (weight) and shift (bias) parameters

**Formula**: `output = weight * (input - mean) / sqrt(variance + eps) + bias`

**Note**: This example uses inference mode with pre-computed statistics, not training mode.

In [ ]:
# BatchNorm2D
input = np.random.rand(1, 3, 128, 128).astype(bfloat16)
input = np.zeros_like(input)  # Use zeros to avoid NaNs in BatchNorm
input_torch = torch.from_numpy(input.astype(np.float32)).to(torch.bfloat16)

means = np.random.randn(input.shape[1]).astype(bfloat16)
bias = np.random.rand(input.shape[1]).astype(bfloat16)
var = np.random.rand(input.shape[1]).astype(bfloat16)
weights = np.random.rand(input.shape[1]).astype(bfloat16)
eps = 1e-5

r = q_ai.batchnorm2d_fprop(
    input, means=means, bias=bias, variances=var, eps=eps, weights=weights
)


means_torch = torch.from_numpy(means.astype(np.float32)).to(torch.bfloat16)
bias_torch = torch.from_numpy(bias.astype(np.float32)).to(torch.bfloat16)
variances_torch = torch.from_numpy(var.astype(np.float32)).to(torch.bfloat16)
weights_torch = torch.from_numpy(weights.astype(np.float32)).to(torch.bfloat16)

if len(input.shape) == 3:
    r_torch = torch.nn.functional.batch_norm(
        input_torch[torch.newaxis, ...],
        means_torch,
        variances_torch,
        weights_torch,
        bias_torch,
        training=False,
        momentum=0.0,
        eps=eps,
    )[0]
else:
    r_torch = torch.nn.functional.batch_norm(
        input_torch,
        means_torch,
        variances_torch,
        weights_torch,
        bias_torch,
        training=False,
        momentum=0.0,
        eps=eps,
    )

r_torch = r_torch.to(torch.float32).numpy().astype(bfloat16)

err = print_errors(r, r_torch)
error_dict_for_test["batchnorm2d_fprop"] = err

## Summary

This notebook demonstrated the core operations available in the Q.ANT Native Computing Toolkit Python API:

✅ **Mathematical operations**: Element-wise multiplication, scaled periodic functions  
✅ **Neural network layers**: Linear, convolution, transposed convolution  
✅ **Activation functions**: ReLU, Sigmoid, Softmax  
✅ **Pooling operations**: Max pooling, average pooling, adaptive average pooling  
✅ **Normalization**: Batch normalization  
✅ **Auxiliary operations**: Bias addition

All operations were validated against PyTorch implementations to ensure correctness.

### Next Steps

To continue learning about the Q.ANT SDK, explore:
- **Digit Recognition**: Understand classification tasks with convolutional networks
- **Image Recognition**: See how these operations combine to build complete neural networks
- **Image Segmentation**: Learn about encoder-decoder architectures
- **Performance Measurement**: Benchmark operations on Q.ANT hardware

### Error Dictionary for Testing

The `error_dict_for_test` dictionary contains all computed errors and can be used for automated validation:

In [ ]:
# Display summary of all operations tested
print(f"Total operations tested: {len(error_dict_for_test)}")
print("\nOperations validated:")
for operation in error_dict_for_test:
    print(f"  ✓ {operation}")